# Prompt Optimization Techniques

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/prompt-engineering/prompt-optimization-techniques.ipynb)

## Overview

This tutorial explores advanced techniques for optimizing prompts when working with large language models. We focus on two key strategies: A/B testing prompts and iterative refinement. These methods are crucial for improving the effectiveness and efficiency of AI-driven applications.

## Motivation

As AI language models become more sophisticated, the quality of prompts used to interact with them becomes increasingly important. Optimized prompts can lead to more accurate, relevant, and useful responses, enhancing the overall performance of AI applications. This tutorial aims to equip learners with practical techniques to systematically improve their prompts.

## Key Components

1. **A/B Testing Prompts**: A method to compare the effectiveness of different prompt variations.
2. **Iterative Refinement**: A strategy for gradually improving prompts based on feedback and results.
3. **Performance Metrics**: Ways to measure and compare the quality of responses from different prompts.
4. **Practical Implementation**: Hands-on examples using HuggingFace Transformers with an open-source model running locally in Google Colab.

## Method Details

1. **Setup**: We'll start by setting up our Google Colab environment with the necessary libraries and loading a quantized open-source model.

2. **A/B Testing**: 
   - Define multiple versions of a prompt
   - Generate responses for each version
   - Compare results using predefined metrics

3. **Iterative Refinement**:
   - Start with an initial prompt
   - Generate responses and evaluate
   - Identify areas for improvement
   - Refine the prompt based on insights
   - Repeat the process to continuously enhance the prompt

4. **Performance Evaluation**:
   - Define relevant metrics (e.g., relevance, specificity, coherence)
   - Implement scoring functions
   - Compare scores across different prompt versions

Throughout the tutorial, we'll use practical examples to demonstrate these techniques, providing learners with hands-on experience in prompt optimization.

## Conclusion

By the end of this tutorial, learners will have gained:
1. Practical skills in conducting A/B tests for prompt optimization
2. Understanding of iterative refinement processes for prompts
3. Ability to define and use metrics for evaluating prompt effectiveness
4. Hands-on experience with HuggingFace Transformers for prompt optimization using a local open-source model

These skills will enable learners to create more effective AI applications by systematically improving their interaction with language models.

## Statistical Foundations of Prompt Testing

### Why One Example Is Never Enough

A critical mistake in prompt engineering is evaluating a prompt with just one or two examples. LLMs are **stochastic systems** — even with the same prompt and input, different runs can produce substantially different outputs (unless temperature is set to exactly 0, and even then, sampling artifacts can cause variation).

This has profound implications:

- **A prompt that looks great on one run might fail on the next.** You may be observing random luck, not genuine quality.
- **Comparing two prompts with one example each is statistically meaningless.** Prompt A beating Prompt B once tells you almost nothing about which is truly better.
- **Small evaluation sets amplify noise.** With n=1, your confidence interval spans nearly the entire quality range.

The solution is borrowed from experimental science: **run multiple trials and apply statistical tests.** This section builds the intuition and tools for doing this rigorously.

> **The n=1 Fallacy**: Drawing conclusions from a single LLM output is like flipping a coin once and concluding it's a two-headed coin. The variance in LLM outputs is real and often surprisingly large.

In [ ]:
# --- Demonstrating LLM Output Variance ---
# Run the SAME prompt 10 times and measure how much the output varies.
# This reveals the stochastic nature of LLM generation.

from collections import Counter

test_prompt = "Explain what a neural network is in exactly two sentences."
num_runs = 10

responses = []
word_counts = []
token_counts = []

print(f"Running the same prompt {num_runs} times...\n")
for i in range(num_runs):
    messages = [{"role": "user", "content": test_prompt}]
    resp = generate(messages, max_new_tokens=256, temperature=0.7)
    responses.append(resp)
    wc = len(resp.split())
    tc = count_tokens(resp)
    word_counts.append(wc)
    token_counts.append(tc)
    print(f"Run {i+1}: {wc} words, {tc} tokens")

# Statistical summary
print(f"\n--- Output Variance Analysis ---")
print(f"Word count — Mean: {np.mean(word_counts):.1f}, Std: {np.std(word_counts):.1f}, "
      f"Min: {min(word_counts)}, Max: {max(word_counts)}")
print(f"Token count — Mean: {np.mean(token_counts):.1f}, Std: {np.std(token_counts):.1f}, "
      f"Min: {min(token_counts)}, Max: {max(token_counts)}")
print(f"Coefficient of variation (tokens): {np.std(token_counts)/np.mean(token_counts)*100:.1f}%")

# Check how many unique responses we got
# Normalize by stripping whitespace for comparison
unique_responses = len(set(r.strip().lower() for r in responses))
print(f"\nUnique responses out of {num_runs}: {unique_responses}")
print(f"This means {unique_responses/num_runs*100:.0f}% of outputs were distinct — the model is NOT deterministic.")

# Show a few example responses to illustrate the variation
print("\n--- Sample Responses (first 3) ---")
for i, resp in enumerate(responses[:3]):
    print(f"\nRun {i+1}: {resp[:200]}...")

In [ ]:
# --- Simple A/B Test with Statistical Significance ---
# Compare two prompt variants with N runs each, then assess significance.

def score_response_quick(response, criterion="clarity and completeness"):

    """Score a single response on a 1-10 scale using the LLM as judge."""
    messages = [
        {"role": "system", "content": "You are a strict evaluator. Reply with ONLY a single integer from 1 to 10."},
        {"role": "user", "content": f"Rate this response on {criterion} (1-10):\n\n{response}"},
    ]
    raw = generate(messages, max_new_tokens=16, temperature=0.0, do_sample=False)
    candidates = re.findall(r"\b(\d{1,2})\b", raw)
    for c in candidates:
        val = int(c)
        if 1 <= val <= 10:
            return val
    return 5  # default if parsing fails

def statistical_ab_test(prompt_a_text, prompt_b_text, n_runs=8):
    """Run an A/B test with n_runs per prompt and compute basic statistics."""
    scores_a, scores_b = [], []

    for i in range(n_runs):
        # Alternate order to reduce temporal bias
        if i % 2 == 0:
            first, second = ("A", prompt_a_text), ("B", prompt_b_text)
        else:
            first, second = ("B", prompt_b_text), ("A", prompt_a_text)

        for label, prompt_text in [first, second]:
            resp = generate([{"role": "user", "content": prompt_text}], max_new_tokens=256)
            score = score_response_quick(resp)
            if label == "A":
                scores_a.append(score)
            else:
                scores_b.append(score)
        print(f"  Trial {i+1}/{n_runs} — A: {scores_a[-1]}, B: {scores_b[-1]}")

    mean_a, mean_b = np.mean(scores_a), np.mean(scores_b)
    std_a, std_b = np.std(scores_a, ddof=1), np.std(scores_b, ddof=1)

    # Simple two-sample t-test (Welch's)
    se = np.sqrt(std_a**2 / n_runs + std_b**2 / n_runs)
    if se > 0:
        t_stat = (mean_a - mean_b) / se
        # Approximate p-value using normal distribution (valid for n >= 8)
        # For a proper test, you'd use scipy.stats — this is a teaching approximation
        from math import erfc, sqrt
        p_value = erfc(abs(t_stat) / sqrt(2))  # two-tailed
    else:
        t_stat, p_value = 0.0, 1.0

    print(f"\n--- A/B Test Results (n={n_runs} per variant) ---")
    print(f"Prompt A — Mean: {mean_a:.2f}, Std: {std_a:.2f}, Scores: {scores_a}")
    print(f"Prompt B — Mean: {mean_b:.2f}, Std: {std_b:.2f}, Scores: {scores_b}")
    print(f"Difference: {mean_a - mean_b:+.2f}")
    print(f"t-statistic: {t_stat:.3f}, p-value: {p_value:.4f}")
    if p_value < 0.05:
        winner = "A" if mean_a > mean_b else "B"
        print(f"✓ Result is STATISTICALLY SIGNIFICANT (p < 0.05). Prompt {winner} is better.")
    else:
        print(f"✗ Result is NOT statistically significant (p = {p_value:.3f}). Need more data or prompts are equivalent.")
    return {"scores_a": scores_a, "scores_b": scores_b, "p_value": p_value}

# Test: concise vs. detailed prompt for the same task
prompt_concise = "What is gradient descent?"
prompt_detailed = "Explain gradient descent step by step, including the intuition, the math, and a concrete example."

print("Running statistical A/B test...\n")
ab_results = statistical_ab_test(prompt_concise, prompt_detailed, n_runs=8)

### Minimum Sample Sizes and When to Trust Comparisons

| Sample Size (per variant) | What You Can Detect | Confidence Level |
|:---:|:---|:---|
| n = 1 | Nothing — pure noise | ❌ Unreliable |
| n = 3–5 | Large differences (3+ points on 10-point scale) | ⚠️ Low |
| n = 8–15 | Moderate differences (1–2 points) | ✓ Moderate |
| n = 20–30 | Small differences (< 1 point) | ✓✓ Good |
| n = 50+ | Subtle differences, interaction effects | ✓✓✓ High |

**Practical guidelines:**

1. **Never trust n=1.** Even n=3 is barely useful. Start at n=8 for meaningful comparisons.
2. **Look at variance, not just means.** If Prompt A scores [9, 3, 8, 2], its mean of 5.5 hides dangerous inconsistency.
3. **Use paired comparisons when possible.** Give both prompts the *same* input and compare side-by-side. This controls for input difficulty.
4. **Watch for evaluation noise.** The LLM judge itself is stochastic — evaluation scores have their own variance layered on top of generation variance.
5. **The cost of being wrong is asymmetric.** Deploying a worse prompt costs ongoing quality; running 10 more test cases costs a few minutes of compute.

## Setup

First, let's install the required packages, load a quantized open-source model (Qwen3-8B) on Google Colab, and define helper functions for generation and token counting.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import re
import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def count_tokens(text):
    """Count tokens in a string using the model's tokenizer."""
    return len(tokenizer.encode(text))

## A/B Testing Prompts

Let's start with A/B testing by comparing different prompt variations for a specific task.

In [ ]:
# Define prompt variations as plain strings with a {topic} placeholder
prompt_a = "Explain {topic} in simple terms."
prompt_b = "Provide a beginner-friendly explanation of {topic}, including key concepts and an example."

def evaluate_response(response, criteria):
    """Evaluate the quality of a response based on given criteria.

    Args:
        response (str): The generated response.
        criteria (list): List of criteria to evaluate.

    Returns:
        float: The average score across all criteria.
    """
    scores = []
    for criterion in criteria:
        print(f"Evaluating response based on {criterion}...")
        messages = [
            {"role": "system", "content": "You are a strict evaluator. Reply with ONLY a single integer from 1 to 10, nothing else."},
            {"role": "user", "content": f"Rate the following response on {criterion} (1-10):\n\n{response}"},
        ]
        eval_response = generate(messages, max_new_tokens=16, temperature=0.0, do_sample=False)
        # Robust score extraction: find all integers and take the first valid one in 1-10
        candidates = re.findall(r"\b(\d{1,2})\b", eval_response)
        score = None
        for c in candidates:
            val = int(c)
            if 1 <= val <= 10:
                score = val
                break
        if score is None:
            print(f"Warning: Could not extract numeric score for {criterion}. Using default score of 5.")
            score = 5
        scores.append(score)
    return np.mean(scores)

def ab_test(prompt_a, prompt_b, query, num_trials=3):
    """A/B test two prompts on the same query."""
    results = {"A": [], "B": []}
    for trial in range(num_trials):
        for label, prompt in [("A", prompt_a), ("B", prompt_b)]:
            messages = [{"role": "system", "content": prompt}, {"role": "user", "content": query}]
            response = generate(messages)
            results[label].append({
                "response": response,
                "tokens": count_tokens(response),
                "words": len(response.split()),
            })
    for label in ["A", "B"]:
        avg_tokens = np.mean([r["tokens"] for r in results[label]])
        avg_words = np.mean([r["words"] for r in results[label]])
        print(f"Prompt {label}: avg {avg_tokens:.0f} tokens, {avg_words:.0f} words")
    return results

# Perform A/B test
topic = "machine learning"
messages_a = [{"role": "user", "content": prompt_a.format(topic=topic)}]
messages_b = [{"role": "user", "content": prompt_b.format(topic=topic)}]
response_a = generate(messages_a)
response_b = generate(messages_b)

criteria = ["clarity", "informativeness", "engagement"]
score_a = evaluate_response(response_a, criteria)
score_b = evaluate_response(response_b, criteria)

print(f"Prompt A score: {score_a:.2f}")
print(f"Prompt B score: {score_b:.2f}")
print(f"Winning prompt: {'A' if score_a > score_b else 'B'}")

# Run the A/B test framework with token counting
print("\nA/B test with token counting:")
ab_results = ab_test(prompt_a.format(topic=topic), prompt_b.format(topic=topic), topic)

## The Evaluation Problem: LLM-as-Judge Biases

### Hidden Biases in Automated Evaluation

Using an LLM to judge the quality of LLM outputs (sometimes called "LLM-as-a-judge") is convenient but introduces **systematic biases** that can silently corrupt your prompt optimization results. Research has documented several:

#### 1. Position Bias
When asked "Which is better, Response A or Response B?", LLMs tend to **prefer whichever response is presented first** (or sometimes last, depending on the model). This means your A/B test results can be an artifact of presentation order, not actual quality.

#### 2. Length Bias (Verbosity Preference)
LLMs systematically prefer **longer responses**, even when the additional length adds no information. A concise, accurate answer may score lower than a verbose, padded one simply because "more text = more effort = better quality" is baked into training data.

#### 3. Self-Preference Bias
An LLM tends to rate outputs generated by the **same model (or similar models)** higher than outputs from different architectures. This creates a feedback loop where optimization converges toward the evaluator's style rather than objective quality.

#### 4. Format Bias
Responses with bullet points, headers, or structured formatting are often rated higher than equivalent prose, regardless of content quality.

> ⚠️ **These biases can completely invalidate your optimization results.** A prompt that "wins" your A/B test might only be winning because it produces longer outputs that the judge prefers, not because it's actually better.

In [ ]:
# --- Demonstrating Position Bias ---
# Present the SAME two responses in different orders and see if the "winner" changes.

# Two responses of similar quality on the same topic
response_x = ("Neural networks learn by adjusting connection weights through backpropagation. "
              "Training data flows forward through layers, errors are computed at the output, "
              "and gradients flow backward to update each weight proportionally to its "
              "contribution to the error.")

response_y = ("A neural network is a computational model inspired by biological neurons. "
              "It processes data through interconnected nodes organized in layers, where each "
              "connection has a learnable weight that is optimized during training using "
              "gradient-based methods.")

def judge_ab(resp_first, resp_second, label_first="A", label_second="B"):

    """Ask the model to pick the better response."""
    messages = [
        {"role": "system", "content": "Compare the two responses and reply with ONLY 'A' or 'B' to indicate which is better."},
        {"role": "user", "content": f"Response A:\n{resp_first}\n\nResponse B:\n{resp_second}\n\nWhich response is better? Reply with only A or B."},
    ]
    result = generate(messages, max_new_tokens=8, temperature=0.0, do_sample=False)
    # Extract A or B from the response
    result_clean = result.strip().upper()
    if "A" in result_clean and "B" not in result_clean:
        return label_first
    elif "B" in result_clean and "A" not in result_clean:
        return label_second
    # If ambiguous, check which letter appears first
    idx_a = result_clean.find("A")
    idx_b = result_clean.find("B")
    if idx_a >= 0 and (idx_b < 0 or idx_a < idx_b):
        return label_first
    return label_second

# Run the comparison multiple times in both orders
n_trials = 6
order1_wins = {"X": 0, "Y": 0}  # X shown first
order2_wins = {"X": 0, "Y": 0}  # Y shown first

print("Position Bias Test: Same responses, different presentation orders\n")

print("Order 1: Response X first, Response Y second")
for i in range(n_trials):
    winner = judge_ab(response_x, response_y, "X", "Y")
    order1_wins[winner] += 1
    print(f"  Trial {i+1}: Winner = {winner}")

print(f"\nOrder 2: Response Y first, Response X second")
for i in range(n_trials):
    winner = judge_ab(response_y, response_x, "Y", "X")
    order2_wins[winner] += 1
    print(f"  Trial {i+1}: Winner = {winner}")

print(f"\n--- Position Bias Results ---")
print(f"When X is shown first: X wins {order1_wins['X']}/{n_trials}, Y wins {order1_wins['Y']}/{n_trials}")
print(f"When Y is shown first: X wins {order2_wins['X']}/{n_trials}, Y wins {order2_wins['Y']}/{n_trials}")

first_position_wins = order1_wins["X"] + order2_wins["Y"]  # times the first-shown won
total = 2 * n_trials
print(f"\nFirst-position win rate: {first_position_wins}/{total} ({first_position_wins/total*100:.0f}%)")
if first_position_wins / total > 0.6:
    print("⚠️  POSITION BIAS DETECTED — the first-shown response wins disproportionately.")
else:
    print("✓ Position bias appears minimal in this test.")

In [ ]:
# --- Demonstrating Length Bias ---
# Compare a concise accurate response with a verbose equivalent.

concise_answer = ("Gradient descent is an optimization algorithm that iteratively adjusts "
                  "parameters in the direction that reduces the loss function, using the "
                  "gradient (partial derivatives) to determine step direction and size.")

verbose_answer = ("Gradient descent is a fundamental and widely-used optimization algorithm "
                  "in machine learning and deep learning. It works by iteratively adjusting "
                  "the parameters of a model in the direction that reduces the loss function. "
                  "The algorithm computes the gradient, which consists of partial derivatives "
                  "of the loss with respect to each parameter, to determine both the direction "
                  "and the relative magnitude of each step. This process is repeated over many "
                  "iterations, gradually moving the parameters toward values that minimize the "
                  "loss. The learning rate hyperparameter controls the step size, balancing "
                  "between convergence speed and stability. It is important to note that "
                  "gradient descent finds local minima, which in practice often work well "
                  "for deep learning due to the high-dimensional nature of the loss landscape.")

print(f"Concise answer: {count_tokens(concise_answer)} tokens")
print(f"Verbose answer: {count_tokens(verbose_answer)} tokens")
print(f"Verbose/Concise ratio: {count_tokens(verbose_answer)/count_tokens(concise_answer):.1f}x\n")

# Score both on the same criteria
criteria_list = ["accuracy", "completeness", "clarity"]
concise_scores = []
verbose_scores = []

for criterion in criteria_list:
    cs = score_response_quick(concise_answer, criterion)
    vs = score_response_quick(verbose_answer, criterion)
    concise_scores.append(cs)
    verbose_scores.append(vs)
    print(f"{criterion:15s} — Concise: {cs}/10, Verbose: {vs}/10")

print(f"\nOverall — Concise: {np.mean(concise_scores):.1f}, Verbose: {np.mean(verbose_scores):.1f}")

# Now ask for a head-to-head comparison
head_to_head = judge_ab(concise_answer, verbose_answer, "Concise", "Verbose")
print(f"\nHead-to-head winner: {head_to_head}")

if head_to_head == "Verbose" or np.mean(verbose_scores) > np.mean(concise_scores):
    print("⚠️  LENGTH BIAS LIKELY — the verbose response scored higher despite containing "
          "essentially the same information. In a real evaluation, you would want to "
          "normalize for length or use length-controlled comparisons.")
else:
    print("✓ The judge preferred the concise response — length bias not dominant here.")

### Mitigating Evaluation Biases

Now that we've seen these biases in action, here are practical strategies to reduce their impact:

| Bias | Mitigation Strategy |
|:---|:---|
| **Position bias** | Randomize presentation order; run each comparison in both orders and average |
| **Length bias** | Normalize scores by response length; add "prefer concise answers" to judge prompt; use length-controlled evaluation |
| **Self-preference** | Use a different model for evaluation than for generation; use multiple judge models |
| **Format bias** | Strip formatting before evaluation; or evaluate content separately from presentation |

**Best practices for LLM-as-Judge evaluation:**

1. **Always randomize order** in pairwise comparisons — never consistently put one variant first.
2. **Use rubric-based scoring** (rate on specific criteria like accuracy, completeness) rather than open-ended "which is better?" questions.
3. **Cross-validate with different judge models** when the stakes are high.
4. **Include calibration examples** — responses where the "right" answer is known — to check judge reliability.
5. **Report evaluation variance**, not just means. If your judge gives the same response scores of 4, 7, and 9 across three runs, your evaluation process needs fixing before your prompts do.

## Iterative Refinement

Now, let's demonstrate the iterative refinement process for improving a prompt.

In [ ]:
def refine_prompt(initial_prompt, topic, iterations=3):
    """Refine a prompt through multiple iterations.

    Args:
        initial_prompt (str): The starting prompt template string (with {topic} placeholder).
        topic (str): The topic to explain.
        iterations (int): Number of refinement iterations.

    Returns:
        str: The final refined prompt template.
    """
    current_prompt = initial_prompt
    for i in range(iterations):
        # Generate a response using the current prompt
        filled = current_prompt.format(topic=topic)
        messages = [{"role": "user", "content": filled}]
        response = generate(messages)

        # Generate feedback and suggestions for improvement
        feedback_messages = [
            {"role": "user", "content": f"Analyze the following explanation of {topic} and suggest improvements to the prompt that generated it:\n\n{response}"},
        ]
        feedback = generate(feedback_messages)

        # Use the feedback to refine the prompt
        refine_messages = [
            {"role": "system", "content": "You are a prompt-engineering assistant. Return ONLY the improved prompt template. Use {topic} as the sole placeholder."},
            {"role": "user", "content": f"Based on this feedback: '{feedback}', improve the following prompt template. Ensure to only use the variable {{topic}} in your template:\n\n{current_prompt}"},
        ]
        refined_template = generate(refine_messages)

        current_prompt = refined_template
        token_count = count_tokens(current_prompt)
        print(f"Iteration {i+1} ({token_count} tokens): {current_prompt[:120]}...")

    return current_prompt

# Perform A/B test to pick the better starting prompt
topic = "machine learning"
messages_a = [{"role": "user", "content": prompt_a.format(topic=topic)}]
messages_b = [{"role": "user", "content": prompt_b.format(topic=topic)}]
response_a = generate(messages_a)
response_b = generate(messages_b)

criteria = ["clarity", "informativeness", "engagement"]
score_a = evaluate_response(response_a, criteria)
score_b = evaluate_response(response_b, criteria)

print(f"Prompt A score: {score_a:.2f}")
print(f"Prompt B score: {score_b:.2f}")
print(f"Winning prompt: {'A' if score_a > score_b else 'B'}")

# Start with the winning prompt and refine iteratively
initial_prompt = prompt_b if score_b > score_a else prompt_a
refined = refine_prompt(initial_prompt, "machine learning")

print("\nFinal refined prompt:")
print(refined)

In [ ]:
# --- Automated Prompt Optimization Loop ---
# This implements the core optimization loop from scratch:
# 1. Start with a baseline prompt
# 2. Generate N responses, evaluate each
# 3. Analyze weaknesses
# 4. Generate an improved prompt variant
# 5. Repeat for K iterations

def automated_prompt_optimization(seed_prompt, test_inputs, n_responses=3, k_iterations=3):
    """
    Optimize a prompt through automated iteration.

    Args:
        seed_prompt: Initial system prompt to optimize.
        test_inputs: List of user inputs to test the prompt against.
        n_responses: Number of responses to generate per input per iteration.
        k_iterations: Number of optimization iterations.

    Returns:
        List of dicts with iteration history.
    """
    history = []
    current_prompt = seed_prompt

    for iteration in range(k_iterations):
        print(f"\n{"="*60}")
        print(f"ITERATION {iteration + 1}/{k_iterations}")
        print(f"{"="*60}")
        print(f"Current prompt ({count_tokens(current_prompt)} tokens):")
        print(f"  {current_prompt[:150]}...")

        # Step 1: Generate and evaluate responses across test inputs
        all_scores = []
        all_responses = []
        weaknesses = []

        for inp in test_inputs:
            scores_for_input = []
            for _ in range(n_responses):
                messages = [
                    {"role": "system", "content": current_prompt},
                    {"role": "user", "content": inp},
                ]
                resp = generate(messages, max_new_tokens=256)
                score = score_response_quick(resp)
                scores_for_input.append(score)
                all_responses.append({"input": inp, "response": resp, "score": score})

            avg = np.mean(scores_for_input)
            all_scores.extend(scores_for_input)
            print(f"  Input: '{inp[:50]}...' — Scores: {scores_for_input}, Avg: {avg:.1f}")

            # Track low-scoring responses as weaknesses
            for s, idx in zip(scores_for_input, range(len(scores_for_input))):
                if s <= 6:
                    weaknesses.append(f"Low score ({s}) on input '{inp[:40]}...'")

        iter_mean = np.mean(all_scores)
        iter_std = np.std(all_scores)
        print(f"\n  Iteration {iteration+1} summary: Mean={iter_mean:.2f}, Std={iter_std:.2f}")

        # Step 2: Analyze weaknesses and generate improved prompt
        weakness_summary = "; ".join(weaknesses[:5]) if weaknesses else "No major weaknesses detected"

        improve_messages = [
            {"role": "system", "content": (
                "You are a prompt-engineering expert. Given a system prompt and its weaknesses, "
                "produce an IMPROVED version of the system prompt. Return ONLY the improved prompt text, "
                "nothing else. Keep the prompt focused and under 100 words."
            )},
            {"role": "user", "content": (
                f"Current system prompt:\n{current_prompt}\n\n"
                f"Mean quality score: {iter_mean:.1f}/10\n"
                f"Weaknesses identified: {weakness_summary}\n\n"
                f"Generate an improved version of this system prompt."
            )},
        ]
        improved_prompt = generate(improve_messages, max_new_tokens=256)

        history.append({
            "iteration": iteration + 1,
            "prompt": current_prompt,
            "mean_score": float(iter_mean),
            "std_score": float(iter_std),
            "n_weaknesses": len(weaknesses),
            "prompt_tokens": count_tokens(current_prompt),
        })

        current_prompt = improved_prompt

    # Final summary
    print(f"\n{"="*60}")
    print("OPTIMIZATION COMPLETE — Iteration Summary")
    print(f"{"="*60}")
    for h in history:
        print(f"  Iter {h['iteration']}: Mean={h['mean_score']:.2f}, Std={h['std_score']:.2f}, "
              f"Prompt tokens={h['prompt_tokens']}, Weaknesses={h['n_weaknesses']}")

    if len(history) >= 2:
        delta = history[-1]["mean_score"] - history[0]["mean_score"]
        print(f"\n  Overall improvement: {delta:+.2f} points over {k_iterations} iterations")

    return history, current_prompt

# Run the optimizer on a real task
seed_prompt = "You are a helpful tutor. Explain concepts clearly."
test_inputs = [
    "What is overfitting in machine learning?",
    "Explain the difference between supervised and unsupervised learning.",
    "What is a loss function and why does it matter?",
]

print("Starting automated prompt optimization...\n")
opt_history, final_prompt = automated_prompt_optimization(
    seed_prompt, test_inputs, n_responses=2, k_iterations=3
)

print(f"\nFinal optimized prompt:\n{final_prompt}")

### How This Relates to Research Tools

The optimization loop above is the **fundamental algorithm** behind several research tools and frameworks:

- **DSPy** (Khattab et al., 2023): Treats prompts as programs with optimizable modules. Uses a similar evaluate → analyze → improve loop but with typed signatures and automated bootstrapping of few-shot examples.
- **PromptBreeder** (Fernando et al., 2023): Evolves prompts using mutation operators inspired by genetic algorithms. Our "analyze weaknesses → generate improved prompt" step is analogous to their mutation operation.
- **OPRO** (Yang et al., 2023): Uses the LLM itself as the optimizer — exactly what we did when we asked the model to improve its own system prompt based on weaknesses.
- **APE** (Automatic Prompt Engineer, Zhou et al., 2022): Generates prompt candidates and selects the best via evaluation — the generate-and-test paradigm.

The key insight across all these approaches: **prompt optimization is search in a discrete space**, and the search can be automated. What differs between methods is:

1. **How candidates are generated** (random mutation, LLM-guided refinement, gradient-based approximation)
2. **How candidates are evaluated** (automated metrics, LLM-as-judge, human feedback)
3. **How the search is guided** (hill climbing, evolutionary strategies, Bayesian optimization)

Our manual implementation above uses the simplest form: greedy hill climbing with LLM-guided refinement. It works surprisingly well for many practical tasks.

## Comparing Original and Refined Prompts

Let's compare the performance of the original and refined prompts.

In [ ]:
# Generate responses from both the original and refined prompts
original_messages = [{"role": "user", "content": initial_prompt.format(topic="machine learning")}]
refined_messages = [{"role": "user", "content": refined.format(topic="machine learning")}]

original_response = generate(original_messages)
refined_response = generate(refined_messages)

print(f"Original response tokens: {count_tokens(original_response)}")
print(f"Refined response tokens: {count_tokens(refined_response)}")

original_score = evaluate_response(original_response, criteria)
refined_score = evaluate_response(refined_response, criteria)

print(f"Original prompt score: {original_score:.2f}")
print(f"Refined prompt score: {refined_score:.2f}")
print(f"Improvement: {(refined_score - original_score):.2f} points")

## Token Efficiency as Optimization Target

So far, we've optimized for *output quality*. But in production, there's another dimension that matters enormously: **token efficiency**.

Every token has a cost — both financial (API pricing is per-token) and temporal (more tokens = more latency). A prompt engineer who ignores efficiency is like a software engineer who ignores performance: the code works, but it doesn't ship.

In [ ]:
# --- Same Quality, Fewer Tokens ---
# Demonstrate that concise prompts can match verbose ones in output quality.

# Three prompt variants for the same task, with increasing verbosity
prompt_minimal = "Explain backpropagation."

prompt_moderate = ("Explain backpropagation in machine learning. Cover the key intuition, "
                   "the chain rule, and how gradients flow backward through the network.")

prompt_verbose = ("You are an expert machine learning educator with deep knowledge of neural "
                  "network training algorithms. Please provide a thorough and comprehensive "
                  "explanation of the backpropagation algorithm as used in neural network "
                  "training. Your explanation should cover: (1) the high-level intuition of "
                  "why backpropagation is needed, (2) the mathematical foundation including "
                  "the chain rule of calculus, (3) how gradients flow backward through the "
                  "network layers, (4) the relationship between backpropagation and gradient "
                  "descent, and (5) any practical considerations or common pitfalls.")

prompt_variants = [
    ("Minimal", prompt_minimal),
    ("Moderate", prompt_moderate),
    ("Verbose", prompt_verbose),
]

print("--- Prompt Token Counts ---")
for name, p in prompt_variants:
    print(f"  {name}: {count_tokens(p)} tokens")

print("\n--- Generating & Scoring (3 runs each) ---")
efficiency_data = []

for name, prompt in prompt_variants:
    scores = []
    output_tokens = []
    for run in range(3):
        resp = generate([{"role": "user", "content": prompt}], max_new_tokens=512)
        score = score_response_quick(resp, "accuracy and completeness")
        scores.append(score)
        output_tokens.append(count_tokens(resp))

    prompt_tok = count_tokens(prompt)
    avg_score = np.mean(scores)
    avg_out = np.mean(output_tokens)
    total_tokens = prompt_tok + avg_out
    efficiency = avg_score / total_tokens * 100  # quality points per 100 tokens

    efficiency_data.append({
        "name": name, "prompt_tokens": prompt_tok,
        "avg_output_tokens": avg_out, "avg_score": avg_score,
        "total_tokens": total_tokens, "efficiency": efficiency,
    })

    print(f"\n  {name}:")
    print(f"    Prompt: {prompt_tok} tokens, Avg output: {avg_out:.0f} tokens")
    print(f"    Avg quality: {avg_score:.1f}/10, Scores: {scores}")
    print(f"    Efficiency: {efficiency:.2f} quality points per 100 tokens")

In [ ]:
# --- Token Efficiency Frontier ---
# Plot the tradeoff: tokens-in vs quality-out.
# The "frontier" shows the best quality achievable at each token budget.

print("\n--- Token Efficiency Frontier ---")
print(f"{'Variant':<12} {'Prompt Tok':<12} {'Total Tok':<12} {'Quality':<10} {'Efficiency':<12}")
print("-" * 58)

for d in efficiency_data:
    print(f"{d['name']:<12} {d['prompt_tokens']:<12} {d['total_tokens']:<12.0f} "
          f"{d['avg_score']:<10.1f} {d['efficiency']:<12.2f}")

# Find the most efficient variant
best_eff = max(efficiency_data, key=lambda x: x["efficiency"])
best_qual = max(efficiency_data, key=lambda x: x["avg_score"])

print(f"\nMost efficient: {best_eff['name']} ({best_eff['efficiency']:.2f} quality/100tok)")
print(f"Highest quality: {best_qual['name']} ({best_qual['avg_score']:.1f}/10)")

# Calculate the cost of the verbose approach relative to minimal
if len(efficiency_data) >= 3:
    minimal = efficiency_data[0]
    verbose = efficiency_data[2]
    if verbose["avg_score"] > 0 and minimal["avg_score"] > 0:
        token_ratio = verbose["total_tokens"] / minimal["total_tokens"]
        quality_ratio = verbose["avg_score"] / minimal["avg_score"]
        print(f"\n  Verbose uses {token_ratio:.1f}x more tokens for {quality_ratio:.2f}x the quality.")
        if token_ratio > quality_ratio:
            print(f"  → The verbose prompt is LESS efficient: you pay more tokens per quality point.")
            print(f"  → At scale (1M requests), this means ~{(verbose["total_tokens"]-minimal["total_tokens"])/1000:.0f}K extra tokens per request.")

### Token Efficiency in Production

In production systems, token efficiency directly translates to:

| Factor | Impact of Extra Tokens |
|:---|:---|
| **Cost** | API pricing is per-token. 2x tokens = 2x cost. At scale, this can mean thousands of dollars per month. |
| **Latency** | Each token takes ~20-50ms to generate. A response with 200 extra tokens adds 4-10 seconds. |
| **Context window** | Wasteful prompts consume context budget, leaving less room for conversation history or retrieved documents. |
| **User experience** | Users waiting for verbose preambles before getting useful content will disengage. |

**The 90/50 Rule of Thumb:** A prompt that achieves 90% of the quality at 50% of the token cost is almost always the better production choice. The marginal quality from verbose prompts rarely justifies the cost at scale.

When optimizing prompts, always ask: *"Is this additional instruction earning its tokens?"* If removing a sentence from the prompt doesn't measurably change output quality, remove it.

## Optimization Pitfalls

Prompt optimization, like any optimization process, can go wrong in systematic ways. Understanding these failure modes is essential before deploying "optimized" prompts in production.

In [ ]:
# --- Pitfall 1: Overfitting to Test Cases ---
# A prompt optimized for one specific input may fail on others.
# This is the prompt engineering equivalent of overfitting in ML.

# Prompt specifically tuned to explain sorting algorithms
overfit_prompt = (
    "You are an expert algorithm teacher. When explaining sorting algorithms, "
    "always start with the time complexity, then describe the mechanism using "
    "a deck-of-cards analogy, then provide Python pseudocode. Keep it under 150 words."
)

# Generic well-crafted prompt
general_prompt = (
    "You are a clear and concise CS educator. Explain concepts with an intuitive "
    "analogy followed by the key technical details. Keep responses focused."
)

# Test on the "in-distribution" topic (what the overfit prompt was designed for)
in_dist_input = "Explain bubble sort."

# Test on "out-of-distribution" topics
ood_inputs = [
    "Explain how a hash table handles collisions.",
    "What is the CAP theorem in distributed systems?",
    "Explain the difference between processes and threads.",
]

print("--- Overfitting Test ---\n")

# Score on in-distribution topic
for label, prompt in [("Overfit", overfit_prompt), ("General", general_prompt)]:
    resp = generate([{"role": "system", "content": prompt}, {"role": "user", "content": in_dist_input}], max_new_tokens=300)
    score = score_response_quick(resp)
    print(f"In-distribution ({in_dist_input[:30]}) — {label}: {score}/10")

# Score on out-of-distribution topics
print(f"\nOut-of-distribution topics:")
overfit_scores = []
general_scores = []

for ood_input in ood_inputs:
    for label, prompt, score_list in [
        ("Overfit", overfit_prompt, overfit_scores),
        ("General", general_prompt, general_scores)
    ]:
        resp = generate([{"role": "system", "content": prompt}, {"role": "user", "content": ood_input}], max_new_tokens=300)
        score = score_response_quick(resp)
        score_list.append(score)
        print(f"  {ood_input[:45]:45s} — {label}: {score}/10")

print(f"\nOverfit prompt OOD mean: {np.mean(overfit_scores):.1f}")
print(f"General prompt OOD mean: {np.mean(general_scores):.1f}")

if np.mean(general_scores) > np.mean(overfit_scores):
    print("\n⚠️  The 'optimized' prompt UNDERPERFORMS the general prompt on new topics!")
    print("This is prompt overfitting: tuning for specific test cases at the expense of generality.")
else:
    print("\nThe overfit prompt generalized reasonably well in this case, but the risk remains.")

In [ ]:
# --- Pitfall 2: Metric Gaming ---
# When optimizing for a specific metric, the model may learn to "game" it —
# producing outputs that score high on the metric but are actually poor quality.

# Scenario: We optimize for "informativeness" score
# A gaming prompt stuffs responses with facts/numbers to score high
gaming_prompt = (
    "Pack your response with as many specific facts, statistics, dates, and named "
    "entities as possible. Include numbered lists and technical terminology. "
    "Prioritize density of information above all else."
)

# A quality prompt focuses on genuine understanding
quality_prompt = (
    "Explain the concept clearly so a smart undergraduate would understand it. "
    "Focus on building genuine intuition before introducing technical details."
)

test_question = "What is reinforcement learning?"

print("--- Metric Gaming Test ---\n")

resp_gaming = generate(
    [{"role": "system", "content": gaming_prompt}, {"role": "user", "content": test_question}],
    max_new_tokens=400
)
resp_quality = generate(
    [{"role": "system", "content": quality_prompt}, {"role": "user", "content": test_question}],
    max_new_tokens=400
)

# Score on the metric being "gamed"
gaming_info_score = score_response_quick(resp_gaming, "informativeness and factual density")
quality_info_score = score_response_quick(resp_quality, "informativeness and factual density")

# Score on a metric that captures ACTUAL quality
gaming_understanding_score = score_response_quick(resp_gaming, "clarity and how well it builds genuine understanding")
quality_understanding_score = score_response_quick(resp_quality, "clarity and how well it builds genuine understanding")

print(f"Gaming prompt:")
print(f"  'Informativeness' score: {gaming_info_score}/10 (the optimized metric)")
print(f"  'Understanding' score:   {gaming_understanding_score}/10 (the real goal)")
print(f"\nQuality prompt:")
print(f"  'Informativeness' score: {quality_info_score}/10 (the optimized metric)")
print(f"  'Understanding' score:   {quality_understanding_score}/10 (the real goal)")

print(f"\n--- Analysis ---")
if gaming_info_score >= quality_info_score and gaming_understanding_score < quality_understanding_score:
    print("⚠️  METRIC GAMING DETECTED:")
    print(f"   The gaming prompt wins on 'informativeness' ({gaming_info_score} vs {quality_info_score})")
    print(f"   but loses on actual understanding ({gaming_understanding_score} vs {quality_understanding_score}).")
    print("   This is Goodhart's Law: when a measure becomes a target, it ceases to be a good measure.")
else:
    print("In this run, metric gaming wasn't clearly demonstrated, but the risk is real.")
    print("The key lesson: always evaluate on MULTIPLE metrics, especially ones you\'re NOT optimizing for.")

### Lessons: Avoiding Optimization Pitfalls

**1. Use Diverse Test Sets**
- Never optimize on fewer than 5-10 diverse test inputs.
- Include edge cases, different phrasings, and topics adjacent to your main task.
- Split your test cases into "optimization" and "holdout" sets — if performance drops on the holdout set, you're overfitting.

**2. Use Multiple Metrics**
- Always evaluate on at least 2-3 metrics that capture different quality dimensions.
- Include at least one metric you're NOT optimizing for — this is your canary for metric gaming.
- Goodhart's Law applies to prompt optimization: "When a measure becomes a target, it ceases to be a good measure."

**3. Test Robustness**
- After optimization, test your prompt on inputs it has never seen.
- Try adversarial inputs — queries that might confuse or trick the prompt.
- Check that the prompt works across different phrasings of the same question.

**4. Human Spot-Checks**
- Automated metrics are necessary but not sufficient. Periodically read actual outputs.
- A prompt that scores 8/10 on automated metrics but produces subtly wrong content is worse than one scoring 7/10 with consistently correct content.

> **Remember:** The goal of prompt optimization is to make the AI more useful to real users, not to maximize a score. Keep the end user in mind throughout the process.